# Angular Spectrum Convergence Study

This notebook investigates how RMSE converges for the angular spectrum method as we vary:
1. **N_photons**: Number of Monte Carlo samples
2. **FFT resolution**: K-space sampling resolution (fft_size)

For each combination, we run multiple trials and compute mean ± std of RMSE.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import j1
from monte_carlo.angular_spectrum import AngularSpectrumSimulator
from monte_carlo import metrics

# Set style
sns.set_theme(style="whitegrid", font_scale=1.5)

# Create output directory
import os
output_dir = '../data/convergence'
os.makedirs(output_dir, exist_ok=True)

## Setup Parameters

In [ ]:
# Fixed simulation parameters
wavelength = 0.532  # microns
focal_length = 10.0  # mm
numerical_aperture = 0.1
n_medium = 1.0

# Convergence study parameters
n_photons_list = [1000, 5000, 10000, 50000, 100000, 500000]
fft_sizes = [128, 256, 512, 1024]  # Different k-space resolutions
n_trials = 5  # Number of trials per combination

# Airy parameters
airy_radius = 1.22 * wavelength / numerical_aperture
k_theory = 2 * np.pi * numerical_aperture / wavelength

print(f"Wavelength: {wavelength} μm")
print(f"NA: {numerical_aperture}")
print(f"Focal length: {focal_length} mm")
print(f"First Airy zero: {airy_radius:.4f} μm")
print(f"\nN_photons range: {min(n_photons_list)} to {max(n_photons_list)}")
print(f"FFT sizes: {fft_sizes}")
print(f"Trials per combination: {n_trials}")

## Compute Theoretical Airy Pattern

In [ ]:
# Create theoretical Airy radial profile
plot_range = 2 * airy_radius
r_bins = np.linspace(0, plot_range, 300)
r_centers = (r_bins[:-1] + r_bins[1:]) / 2
bin_areas = np.pi * (r_bins[1:]**2 - r_bins[:-1]**2)

# Theoretical Airy pattern
kr_theory = k_theory * r_centers
airy_theory = np.ones_like(kr_theory)
nonzero = kr_theory != 0
airy_theory[nonzero] = (2 * j1(kr_theory[nonzero]) / kr_theory[nonzero])**2

print(f"Radial bins: {len(r_bins)}")
print(f"Radial range: [0, {plot_range:.4f}] μm")

## Run Convergence Study

In [ ]:
# Storage for results: {fft_size: {n_photons: {'mean': x, 'std': y, 'trials': []}}}
rmse_results = {fft_size: {} for fft_size in fft_sizes}

for fft_size in fft_sizes:
    print(f"\n{'='*80}")
    print(f"FFT Size = {fft_size}")
    print(f"{'='*80}")
    
    for n_photons in n_photons_list:
        print(f"  N_photons = {n_photons}...")
        rmse_trials = []
        
        for trial in range(n_trials):
            # Create simulator with different seed for each trial
            sim = AngularSpectrumSimulator(
                n_photons=n_photons,
                wavelength=wavelength,
                focal_length=focal_length,
                numerical_aperture=numerical_aperture,
                n_medium=n_medium,
                random_seed=trial,
                fft_size=fft_size
            )
            
            # Run simulation
            results = sim.propagate()
            x, y, _ = results['focal_positions']
            
            # Compute radial profile
            r = np.sqrt(x**2 + y**2)
            hist_r, _ = np.histogram(r, bins=r_bins)
            intensity_r = hist_r / bin_areas
            intensity_r = intensity_r / np.max(intensity_r)  # Normalize to peak
            
            # Compute RMSE
            rmse_val = metrics.rmse(intensity_r, airy_theory)
            rmse_trials.append(rmse_val)
        
        # Store results
        rmse_results[fft_size][n_photons] = {
            'mean': np.mean(rmse_trials),
            'std': np.std(rmse_trials),
            'trials': rmse_trials
        }
        
        print(f"    RMSE: {rmse_results[fft_size][n_photons]['mean']:.6f} ± {rmse_results[fft_size][n_photons]['std']:.6f}")

print("\nConvergence study complete!")

## Plot Results

In [ ]:
# Create figure with multiple lines for different FFT sizes
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Color palette
colors = sns.color_palette("husl", len(fft_sizes))

# Plot 1: RMSE vs N_photons for different FFT sizes
ax = axes[0]
for i, fft_size in enumerate(fft_sizes):
    n_photons_array = np.array(n_photons_list)
    rmse_mean = np.array([rmse_results[fft_size][n]['mean'] for n in n_photons_list])
    rmse_std = np.array([rmse_results[fft_size][n]['std'] for n in n_photons_list])
    
    ax.errorbar(n_photons_array, rmse_mean, yerr=rmse_std, 
                marker='o', markersize=8, linewidth=2.5, capsize=4, capthick=1.5,
                label=f'FFT size = {fft_size}', color=colors[i])

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Number of Photons')
ax.set_ylabel('RMSE')
ax.set_title('Angular Spectrum: RMSE vs N_photons', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: RMSE vs FFT size for different N_photons
ax = axes[1]
colors2 = sns.color_palette("viridis", len(n_photons_list))

for i, n_photons in enumerate(n_photons_list):
    fft_array = np.array(fft_sizes)
    rmse_mean = np.array([rmse_results[fft][n_photons]['mean'] for fft in fft_sizes])
    rmse_std = np.array([rmse_results[fft][n_photons]['std'] for fft in fft_sizes])
    
    ax.errorbar(fft_array, rmse_mean, yerr=rmse_std,
                marker='s', markersize=8, linewidth=2.5, capsize=4, capthick=1.5,
                label=f'N = {n_photons}', color=colors2[i])

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('FFT Size (k-space resolution)')
ax.set_ylabel('RMSE')
ax.set_title('Angular Spectrum: RMSE vs FFT Resolution', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{output_dir}/angular_spectrum_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPlot saved to: {output_dir}/angular_spectrum_convergence.png")

## Summary Statistics

In [ ]:
print("\n" + "="*100)
print("ANGULAR SPECTRUM CONVERGENCE STUDY")
print("="*100)

for fft_size in fft_sizes:
    print(f"\nFFT Size = {fft_size}")
    print("-"*100)
    print(f"{'N_photons':<15} {'RMSE (mean)':<20} {'RMSE (std)':<20} {'Relative Std %':<15}")
    print("-"*100)
    for n in n_photons_list:
        mean_rmse = rmse_results[fft_size][n]['mean']
        std_rmse = rmse_results[fft_size][n]['std']
        rel_std = (std_rmse / mean_rmse) * 100 if mean_rmse > 0 else 0
        print(f"{n:<15} {mean_rmse:<20.6f} {std_rmse:<20.6f} {rel_std:<15.2f}")

print("\n" + "="*100)